# Incident Commander — GRPO Training (Multi-turn)

Fine-tunes **Llama-3.2-1B-Instruct** with GRPO on the Incident Commander RL environment hosted on Hugging Face Spaces.

**What changed vs. the earlier notebook**
* Reward functions now run a **multi-turn rollout** per completion — the agent investigates, hypothesises, fixes, monitors, and resolves inside *one* reward call. Without this, `service_recovery`, `root_cause_accuracy`, and `speed` rewards never fire.
* The four reward signals are exposed as **separate `reward_funcs`** so TRL logs each one independently.
* Real before / after evaluation runs with full episodes (not single steps).
* All plots are saved to `assets/` so they can be embedded in the README.

**Hardware:** free-tier T4 (Colab) is enough.

**References**
* [OpenEnv](https://github.com/meta-pytorch/OpenEnv) · [TRL GRPO](https://huggingface.co/docs/trl/main/en/grpo_trainer) · [Unsloth](https://github.com/unslothai/unsloth)

## 1. Install dependencies (pinned)

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q "openenv-core[core]>=0.2.2" "trl>=0.12.0" "datasets>=2.20.0" \
    "peft>=0.11.0" "accelerate>=0.33.0" "bitsandbytes>=0.43.0" \
    matplotlib numpy requests

## 2. Clone repo & verify GPU

In [ ]:
import os, sys, torch

REPO_DIR = '/content/incident_commander'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Unknown-guy-369/Incident_commander.git {REPO_DIR}
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Sanity-check the remote environment

In [ ]:
from rollout import SyncEnvClient, DEFAULT_REMOTE_URL

HF_SPACE_URL = os.environ.get('INCIDENT_COMMANDER_ENV_URL', DEFAULT_REMOTE_URL)
print(f'Using env: {HF_SPACE_URL}')

with SyncEnvClient(HF_SPACE_URL) as env:
    init = env.reset(difficulty=1)
    obs = init['observation']
    print('Alert :', obs.get('alert_summary'))
    print('Services:', [s['name'] for s in obs.get('services_overview', [])])
    step = env.step(action_type='read_logs', target_service='payment-service')
    print('First step reward:', step.get('reward'))
print('Remote env verified.')

## 4. Load Llama-3.2-1B-Instruct via Unsloth (4-bit)

In [ ]:
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL('GRPO', FastLanguageModel)
from trl import GRPOConfig, GRPOTrainer

MODEL_NAME       = 'unsloth/Llama-3.2-1B-Instruct'
MAX_SEQ_LENGTH   = 2048
LORA_RANK        = 16
BATCH_SIZE       = 2          # small batch + multi-turn rollouts means lots of HTTP
GRAD_ACCUM_STEPS = 4
LEARNING_RATE    = 2e-5
NUM_GENERATIONS  = 4          # GRPO group size
ROLLOUT_STEPS    = 12         # max env steps inside each rollout

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.float16,
    load_in_4bit=True,
    fast_inference=False,   # vLLM crashes on T4 (compute capability 7.5)
    max_lora_rank=LORA_RANK,
    gpu_memory_utilization=0.6,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    lora_alpha=LORA_RANK,
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)
print('Model + LoRA ready.')

## 5. Build rollout-based reward functions

Each completion triggers a *full* multi-turn episode against the remote env. The four reward signals defined in `rewards.py` are exposed as separate functions so TRL logs them independently.

In [ ]:
from rollout import rollout_episode, SYSTEM_PROMPT
from rewards import make_grpo_reward_fns

DIFFICULTY = 1

def make_generate_fn(model, tokenizer, max_new_tokens=180, temperature=0.7):
    device = next(model.parameters()).device
    def _generate(prompt: str) -> str:
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1536).to(device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=temperature > 0,
                temperature=temperature,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )
        return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return _generate

_train_generate_fn = make_generate_fn(model, tokenizer)

def episode_rollout(prompt: str, completion: str):
    """Drives a full multi-turn episode for one (prompt, completion) pair."""
    with SyncEnvClient(HF_SPACE_URL) as env:
        state = rollout_episode(
            env, _train_generate_fn,
            max_steps=ROLLOUT_STEPS, difficulty=DIFFICULTY,
        )
    return state.to_dict()

reward_fns = make_grpo_reward_fns(episode_rollout)
print(f'Wired {len(reward_fns)} reward functions: {[fn.__name__ for fn in reward_fns]}')

## 6. Generate a small prompt dataset

In [ ]:
from datasets import Dataset

def generate_initial_prompts(num_samples=80, difficulty=DIFFICULTY):
    rows = []
    for i in range(num_samples):
        try:
            with SyncEnvClient(HF_SPACE_URL) as env:
                init = env.reset(difficulty=difficulty)
            obs = init.get('observation', {})
            prompt = (
                f"System: {SYSTEM_PROMPT}\n\n"
                f"Initial alert: {obs.get('alert_summary','')}\n"
                f"Services overview: {obs.get('services_overview','')}\n\n"
                'Begin the investigation. What is your first action?'
            )
            rows.append({'prompt': prompt, 'seed': i})
        except Exception as e:
            print(f'  prompt {i} failed: {e}')
        if (i + 1) % 25 == 0:
            print(f'  generated {i + 1}/{num_samples}')
    return Dataset.from_list(rows)

dataset = generate_initial_prompts(num_samples=80)
print('Dataset size:', len(dataset))

## 7. BEFORE-training evaluation (full multi-turn episodes)

In [ ]:
import numpy as np

def evaluate_episodes(model, tokenizer, num_episodes=15, label='model',
                     difficulty=DIFFICULTY, max_steps=ROLLOUT_STEPS, temperature=0.7):
    print(f'\n=== Evaluating: {label} ({num_episodes} episodes) ===')
    FastLanguageModel.for_inference(model)
    gen_fn = make_generate_fn(model, tokenizer, max_new_tokens=180, temperature=temperature)

    rewards, format_scores, valid_actions = [], [], []
    resolved_count = 0
    correct_root_cause = 0
    per_signal = {'service_recovery': [], 'root_cause_accuracy': [],
                  'action_quality': [], 'speed': []}
    for ep in range(num_episodes):
        try:
            with SyncEnvClient(HF_SPACE_URL) as env:
                state = rollout_episode(env, gen_fn, max_steps=max_steps, difficulty=difficulty)
        except Exception as e:
            print(f'  episode {ep} failed: {e}')
            continue
        bd = (state.last_observation or {}).get('reward_breakdown') or {}
        rewards.append(state.last_reward)
        for k in per_signal:
            per_signal[k].append(float(bd.get(k, 0.0)))
        valid_actions.append(int(any(a in {'read_logs','read_metrics','identify_cause'} for a in state.actions_taken)))
        format_scores.append(1.0 if state.actions_taken else 0.0)
        if state.post_fix_status == 'recovered' and 'resolve' in state.actions_taken:
            resolved_count += 1
        if state.locked_hypothesis and state.locked_hypothesis == state.true_root_cause:
            correct_root_cause += 1
        hyp_mark = 'OK' if state.locked_hypothesis == state.true_root_cause else 'X'
        print(f'  ep {ep+1:>2}/{num_episodes}  steps={state.steps_used:>2}  '
              f'reward={state.last_reward:.3f}  status={state.post_fix_status}  '
              f'hyp={hyp_mark}')

    FastLanguageModel.for_training(model)
    summary = {
        'label': label,
        'avg_reward': float(np.mean(rewards)) if rewards else 0.0,
        'std_reward': float(np.std(rewards)) if rewards else 0.0,
        'avg_format': float(np.mean(format_scores)) if format_scores else 0.0,
        'valid_action_rate': float(np.mean(valid_actions)) if valid_actions else 0.0,
        'resolve_rate': resolved_count / max(num_episodes, 1),
        'root_cause_accuracy': correct_root_cause / max(num_episodes, 1),
        'per_signal_mean': {k: float(np.mean(v)) if v else 0.0 for k, v in per_signal.items()},
        'rewards': rewards,
    }
    print(f"  Summary: avg_reward={summary['avg_reward']:.4f}  resolve_rate={summary['resolve_rate']*100:.1f}%  "
          f"root_cause_accuracy={summary['root_cause_accuracy']*100:.1f}%")
    return summary

before = evaluate_episodes(model, tokenizer, num_episodes=15, label='Before GRPO')

## 8. Configure & run GRPO

In [ ]:
training_args = GRPOConfig(
    output_dir='outputs/commander_grpo',
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    max_prompt_length=1024,
    max_completion_length=256,
    num_generations=NUM_GENERATIONS,
    max_steps=200,
    logging_steps=5,
    save_steps=40,
    beta=0.04,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    report_to='none',
)
trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_fns,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)
print('GRPOTrainer ready (max_steps=200).')

In [ ]:
print('Starting GRPO training...')
trainer.train()
print('Training complete.')

## 9. Save the merged model

In [ ]:
import os
os.makedirs('outputs/commander_final', exist_ok=True)
model.save_pretrained_merged('outputs/commander_final', tokenizer, save_method='merged_16bit')
print('Merged 16-bit model saved to outputs/commander_final')

## 10. AFTER-training evaluation

In [ ]:
after = evaluate_episodes(model, tokenizer, num_episodes=15, label='After GRPO')

## 11. Plots & summary table

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

os.makedirs('assets', exist_ok=True)

metrics_names = ['Avg Reward', 'Resolve Rate', 'Root-cause Accuracy']
before_vals = [before['avg_reward'], before['resolve_rate'], before['root_cause_accuracy']]
after_vals  = [after['avg_reward'],  after['resolve_rate'],  after['root_cause_accuracy']]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
x = np.arange(len(metrics_names)); w = 0.35
axes[0].bar(x - w/2, before_vals, w, label='Before',  color='#ff6b6b', alpha=0.85)
axes[0].bar(x + w/2, after_vals,  w, label='After',   color='#51cf66', alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(metrics_names)
axes[0].set_title('Before vs After GRPO', fontweight='bold')
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

axes[1].hist(after['rewards'], bins=10, color='#51cf66', alpha=0.7, edgecolor='black')
axes[1].axvline(before['avg_reward'], color='red',   linestyle='--', linewidth=2, label=f"Before mean: {before['avg_reward']:.3f}")
axes[1].axvline(after['avg_reward'],  color='green', linestyle='--', linewidth=2, label=f"After mean:  {after['avg_reward']:.3f}")
axes[1].set_title('Post-training reward distribution', fontweight='bold')
axes[1].set_xlabel('Reward'); axes[1].set_ylabel('Episodes')
axes[1].legend(); axes[1].grid(alpha=0.3)

deltas = [after_vals[i] - before_vals[i] for i in range(len(metrics_names))]
colors = ['#51cf66' if d >= 0 else '#ff6b6b' for d in deltas]
axes[2].bar(metrics_names, deltas, color=colors, alpha=0.85, edgecolor='black')
axes[2].axhline(0, color='black', linewidth=0.8)
axes[2].set_title('Improvement delta (after - before)', fontweight='bold')
axes[2].grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('assets/before_after_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

fig2, ax2 = plt.subplots(figsize=(10, 5))
signals = list(before['per_signal_mean'].keys())
b_means = [before['per_signal_mean'][k] for k in signals]
a_means = [after['per_signal_mean'][k]  for k in signals]
x = np.arange(len(signals)); w = 0.35
ax2.bar(x - w/2, b_means, w, label='Before', color='#ff6b6b', alpha=0.85)
ax2.bar(x + w/2, a_means, w, label='After',  color='#51cf66', alpha=0.85)
ax2.set_xticks(x); ax2.set_xticklabels(signals, rotation=15, ha='right')
ax2.set_title('Per-signal reward (raw points) — before vs after', fontweight='bold')
ax2.legend(); ax2.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('assets/per_signal_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '=' * 60)
print('Incident Commander - GRPO results')
print('=' * 60)
print(f"{'Metric':<25}{'Before':>10}{'After':>10}{'Delta':>10}")
print('-' * 60)
for name, b, a in zip(metrics_names, before_vals, after_vals):
    delta = a - b
    arrow = 'UP' if delta > 0 else ('DOWN' if delta < 0 else 'FLAT')
    print(f'{name:<25}{b:>10.3f}{a:>10.3f}{delta:>+10.3f} {arrow}')
print('=' * 60)
print('Plots saved to assets/before_after_comparison.png and assets/per_signal_comparison.png')

## 12. (Optional) push merged model to HF Hub

In [ ]:
# Uncomment & set HF_TOKEN to push:
# from huggingface_hub import HfApi
# HfApi().upload_folder(
#     folder_path='outputs/commander_final',
#     repo_id='your-username/incident-commander-llama32-1b',
#     repo_type='model',
# )
print('Model lives at outputs/commander_final.')

---

## Action schema reference

| Action | Target | Extra | Meaning |
|---|---|---|---|
| read_logs | yes | time_range_minutes | Read recent logs for a service |
| read_metrics | yes | — | Check CPU / memory / error rate / p99 |
| read_deployment_history | no | — | List recent deploys |
| read_dependency_graph | no | — | Show service-call graph |
| identify_cause | yes | hypothesis | Lock the root cause hypothesis |
| restart_pod | yes | — | Restart the pod |
| rollback | yes | — | Revert the latest deploy |
| scale_up | yes | — | Add compute |
| hotfix | yes | — | Patch config / cert / pool |
| escalate | no | justification | Page a human |
| monitor_recovery | no | — | Observe post-fix status |
| resolve | no | justification | Close the incident |

## Root cause → fix mapping

| Root cause | Fix |
|---|---|
| memory_limit_too_low | scale_up |
| bad_deployment | rollback |
| connection_pool_exhausted | hotfix |
| traffic_spike | scale_up |
| dependency_failure | restart_pod |
| config_error | hotfix |
| redis_down | restart_pod |
| certificate_expired | hotfix |